# Synthetic Satellite Downlink PSD Dataset Generator (GNURadio)

This notebook generates labeled Power Spectral Density (PSD) plots of synthetic satellite downlink signals using GNURadio.

## Overview
- Generate multi-carrier satellite downlink scenarios
- Support DVB-S2 modulation schemes (QPSK, 8PSK, 16APSK, 32APSK)
- Compute and visualize PSDs with labeled carrier parameters
- Export dataset for ML training

## 1. Imports & Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import json
import os
from datetime import datetime
from pathlib import Path

# GNURadio imports
try:
    from gnuradio import gr
    from gnuradio import blocks
    from gnuradio import analog
    from gnuradio import digital
    from gnuradio import filter as gr_filter
    print("✓ GNURadio imported successfully")
except ImportError as e:
    print(f"✗ GNURadio import error: {e}")
    print("Some functionality may be limited.")

# Set random seed for reproducibility
np.random.seed(42)

# Matplotlib settings
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

✓ GNURadio imported successfully


## 2. Parameter Configuration

In [2]:
# Dataset generation parameters
NUM_SAMPLES = 100  # Total number of PSD samples to generate
MIN_CARRIERS = 1    # Minimum carriers per sample
MAX_CARRIERS = 5    # Maximum carriers per sample

# RF parameters (Ku-band satellite downlink)
SAMPLE_RATE = 100e6  # 100 MHz sampling rate
FREQ_SPAN = 80e6     # 80 MHz frequency span

# Carrier parameter ranges
PARAM_RANGES = {
    'bandwidth_mhz': (1, 36),           # Transponder bandwidth
    'modulation': ['QPSK', '8PSK', '16APSK', '32APSK'],
    'roll_off': [0.20, 0.25, 0.35],     # DVB-S2 standard roll-offs
    'code_rate': ['1/4', '1/3', '2/5', '1/2', '3/5', '2/3', '3/4', '4/5', '5/6', '8/9', '9/10'],
    'power_dbm': (-80, -40),            # Carrier power
    'cn_db': (-2.4, 16.0),              # Carrier-to-noise ratio
}

# Output paths
DATA_DIR = Path('../../data/raw')
PSD_IMG_DIR = DATA_DIR / 'psd_images'
PSD_ARRAY_DIR = DATA_DIR / 'psd_arrays'
LABELS_FILE = DATA_DIR / 'labels.json'

# Create directories
for d in [PSD_IMG_DIR, PSD_ARRAY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Dataset configuration:")
print(f"  Samples: {NUM_SAMPLES}")
print(f"  Carriers per sample: {MIN_CARRIERS}-{MAX_CARRIERS}")
print(f"  Sample rate: {SAMPLE_RATE/1e6:.0f} MHz")
print(f"  Frequency span: {FREQ_SPAN/1e6:.0f} MHz")

Dataset configuration:
  Samples: 100
  Carriers per sample: 1-5
  Sample rate: 100 MHz
  Frequency span: 80 MHz


## 3. Signal Generation Functions (GNURadio)

In [3]:
def modulation_to_constellation(mod_type):
    """Map modulation type to constellation object."""
    # Note: GNURadio 3.10+ uses different constellation objects
    # This is a simplified mapping
    constellations = {
        'QPSK': digital.constellation_qpsk(),
        '8PSK': digital.constellation_8psk(),
    }
    # For APSK, we'd need custom constellation points
    return constellations.get(mod_type, digital.constellation_qpsk())


def generate_carrier_gnuradio(center_freq, bandwidth, modulation, symbol_rate, 
                              roll_off, power_dbm, sample_rate, num_samples):
    """
    Generate a single modulated carrier using GNURadio blocks.
    
    Returns:
        np.array: Complex baseband signal
    """
    # This is a simplified GNURadio flowgraph
    # In practice, you'd build a full flowgraph with tb = gr.top_block()
    
    # For now, generate random symbols and modulate
    sps = int(sample_rate / symbol_rate)  # Samples per symbol
    num_symbols = num_samples // sps
    
    # Generate random symbols (simplified)
    if modulation == 'QPSK':
        symbols = np.random.choice([1+1j, 1-1j, -1+1j, -1-1j], num_symbols)
    elif modulation == '8PSK':
        phases = np.arange(8) * 2 * np.pi / 8
        constellation = np.exp(1j * phases)
        symbols = np.random.choice(constellation, num_symbols)
    else:
        # Fallback to QPSK
        symbols = np.random.choice([1+1j, 1-1j, -1+1j, -1-1j], num_symbols)
    
    # Root-raised cosine pulse shaping
    ntaps = 11 * sps
    rrc_taps = signal.firwin(ntaps, 1.0/sps, window=('kaiser', 5.0))
    
    # Upsample and filter
    upsampled = np.zeros(num_symbols * sps, dtype=complex)
    upsampled[::sps] = symbols
    shaped = signal.lfilter(rrc_taps, 1, upsampled)
    
    # Frequency shift to center frequency
    t = np.arange(len(shaped)) / sample_rate
    carrier_sig = shaped * np.exp(2j * np.pi * center_freq * t)
    
    # Apply power scaling
    power_linear = 10 ** (power_dbm / 10)
    carrier_sig *= np.sqrt(power_linear)
    
    return carrier_sig[:num_samples]


def add_noise(signal_data, cn_db, sample_rate):
    """
    Add AWGN to achieve target C/N ratio.
    """
    signal_power = np.mean(np.abs(signal_data) ** 2)
    cn_linear = 10 ** (cn_db / 10)
    noise_power = signal_power / cn_linear
    
    noise = np.sqrt(noise_power / 2) * (np.random.randn(len(signal_data)) + 
                                         1j * np.random.randn(len(signal_data)))
    return signal_data + noise


def combine_carriers(carrier_params, sample_rate, num_samples):
    """
    Generate and combine multiple carriers.
    
    Args:
        carrier_params: List of dictionaries with carrier parameters
        
    Returns:
        np.array: Combined complex signal
    """
    combined = np.zeros(num_samples, dtype=complex)
    
    for params in carrier_params:
        carrier = generate_carrier_gnuradio(
            center_freq=params['center_freq_hz'],
            bandwidth=params['bandwidth_hz'],
            modulation=params['modulation'],
            symbol_rate=params['symbol_rate_sps'],
            roll_off=params['roll_off'],
            power_dbm=params['power_dbm'],
            sample_rate=sample_rate,
            num_samples=num_samples
        )
        combined += carrier
    
    # Add noise based on average C/N
    avg_cn = np.mean([p['cn_ratio_db'] for p in carrier_params])
    combined = add_noise(combined, avg_cn, sample_rate)
    
    return combined

print("✓ Signal generation functions defined")

✓ Signal generation functions defined


## 4. PSD Computation

In [4]:
def compute_psd(signal_data, sample_rate, nperseg=1024):
    """
    Compute Power Spectral Density using Welch's method.
    
    Returns:
        freqs: Frequency bins (Hz)
        psd_db: PSD in dB scale
    """
    freqs, psd = signal.welch(signal_data, fs=sample_rate, nperseg=nperseg,
                              return_onesided=False, scaling='density')
    
    # Shift to center DC at middle
    freqs = np.fft.fftshift(freqs)
    psd = np.fft.fftshift(psd)
    
    # Convert to dB
    psd_db = 10 * np.log10(psd + 1e-12)  # Add small value to avoid log(0)
    
    return freqs, psd_db

print("✓ PSD computation function defined")

✓ PSD computation function defined


## 5. Visualization

In [5]:
def plot_psd(freqs, psd_db, carrier_params, save_path=None, show=True):
    """
    Plot PSD with carrier annotations.
    """
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot PSD
    ax.plot(freqs / 1e6, psd_db, linewidth=0.8, color='blue', alpha=0.7)
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel('Power Spectral Density (dB/Hz)')
    ax.set_title('Satellite Downlink PSD with Labeled Carriers')
    ax.grid(True, alpha=0.3)
    
    # Annotate carriers
    colors = plt.cm.Set3(np.linspace(0, 1, len(carrier_params)))
    
    for idx, params in enumerate(carrier_params):
        fc = params['center_freq_hz'] / 1e6  # MHz
        bw = params['bandwidth_hz'] / 1e6    # MHz
        
        # Draw carrier span
        ax.axvspan(fc - bw/2, fc + bw/2, alpha=0.2, color=colors[idx])
        
        # Add label
        label = f"C{idx}: {params['modulation']}\n{params['symbol_rate_sps']/1e6:.1f} Msps"
        ax.text(fc, ax.get_ylim()[1] - 5 - idx*3, label, 
                ha='center', fontsize=8, bbox=dict(boxstyle='round', 
                facecolor=colors[idx], alpha=0.5))
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    
    if show:
        plt.show()
    else:
        plt.close()
    
    return fig

print("✓ Visualization function defined")

✓ Visualization function defined


## 6. Carrier Parameter Sampling

In [6]:
def sample_carrier_params(freq_span, sample_rate, carrier_id):
    """
    Sample random carrier parameters from defined ranges.
    """
    # Bandwidth
    bw_mhz = np.random.uniform(*PARAM_RANGES['bandwidth_mhz'])
    bw_hz = bw_mhz * 1e6
    
    # Center frequency (keep away from edges)
    margin = bw_hz
    fc_hz = np.random.uniform(-freq_span/2 + margin, freq_span/2 - margin)
    
    # Modulation and other params
    modulation = np.random.choice(PARAM_RANGES['modulation'])
    roll_off = np.random.choice(PARAM_RANGES['roll_off'])
    code_rate = np.random.choice(PARAM_RANGES['code_rate'])
    power_dbm = np.random.uniform(*PARAM_RANGES['power_dbm'])
    cn_db = np.random.uniform(*PARAM_RANGES['cn_db'])
    
    # Symbol rate from bandwidth and roll-off
    symbol_rate = bw_hz / (1 + roll_off)
    
    return {
        'id': carrier_id,
        'center_freq_hz': fc_hz,
        'bandwidth_hz': bw_hz,
        'modulation': modulation,
        'symbol_rate_sps': symbol_rate,
        'roll_off': roll_off,
        'code_rate': code_rate,
        'power_dbm': power_dbm,
        'cn_ratio_db': cn_db
    }

print("✓ Parameter sampling function defined")

✓ Parameter sampling function defined


## 7. Dataset Generation Loop

In [7]:
def generate_dataset(num_samples, min_carriers, max_carriers):
    """
    Main dataset generation function.
    """
    dataset = []
    num_time_samples = int(SAMPLE_RATE * 0.01)  # 10ms of data
    
    for sample_idx in range(num_samples):
        # Random number of carriers
        num_carriers = np.random.randint(min_carriers, max_carriers + 1)
        
        # Sample carrier parameters
        carriers = [sample_carrier_params(FREQ_SPAN, SAMPLE_RATE, i) 
                    for i in range(num_carriers)]
        
        # Generate combined signal
        signal_data = combine_carriers(carriers, SAMPLE_RATE, num_time_samples)
        
        # Compute PSD
        freqs, psd_db = compute_psd(signal_data, SAMPLE_RATE)
        
        # Save PSD plot
        img_path = PSD_IMG_DIR / f'psd_{sample_idx:05d}.png'
        plot_psd(freqs, psd_db, carriers, save_path=img_path, show=False)
        
        # Save PSD array
        array_path = PSD_ARRAY_DIR / f'psd_{sample_idx:05d}.npy'
        np.save(array_path, {'freqs': freqs, 'psd_db': psd_db})
        
        # Store metadata
        metadata = {
            'sample_id': sample_idx,
            'psd_image': str(img_path.relative_to(DATA_DIR)),
            'psd_array': str(array_path.relative_to(DATA_DIR)),
            'sample_rate': SAMPLE_RATE,
            'frequency_span': FREQ_SPAN,
            'num_carriers': num_carriers,
            'carriers': carriers,
            'generation_timestamp': datetime.now().isoformat(),
            'method': 'gnuradio'
        }
        dataset.append(metadata)
        
        if (sample_idx + 1) % 10 == 0:
            print(f"Generated {sample_idx + 1}/{num_samples} samples")
    
    return dataset

print("✓ Dataset generation function defined")

✓ Dataset generation function defined


## 8. Generate Dataset

In [8]:
print(f"Starting dataset generation...")
print(f"This will generate {NUM_SAMPLES} samples with {MIN_CARRIERS}-{MAX_CARRIERS} carriers each.\n")

dataset = generate_dataset(NUM_SAMPLES, MIN_CARRIERS, MAX_CARRIERS)

print(f"\n✓ Generated {len(dataset)} samples")

Starting dataset generation...
This will generate 100 samples with 1-5 carriers each.



ValueError: operands could not be broadcast together with shapes (1000000,) (999999,) (1000000,) 

## 9. Save Labels

In [ ]:
with open(LABELS_FILE, 'w') as f:
    json.dump(dataset, f, indent=2)

print(f"✓ Saved labels to {LABELS_FILE}")
print(f"  Total size: {LABELS_FILE.stat().st_size / 1024:.1f} KB")

## 10. Validation & Statistics

In [ ]:
# Analyze dataset
import pandas as pd

# Extract carrier statistics
all_carriers = []
for sample in dataset:
    all_carriers.extend(sample['carriers'])

df = pd.DataFrame(all_carriers)

print("Dataset Statistics:")
print(f"  Total samples: {len(dataset)}")
print(f"  Total carriers: {len(all_carriers)}")
print(f"  Avg carriers per sample: {len(all_carriers)/len(dataset):.2f}\n")

print("Modulation distribution:")
print(df['modulation'].value_counts())
print("\nRoll-off distribution:")
print(df['roll_off'].value_counts())
print("\nCode rate distribution:")
print(df['code_rate'].value_counts())

## 11. Visualize Sample PSDs

In [ ]:
# Display a few random examples
sample_indices = np.random.choice(len(dataset), min(3, len(dataset)), replace=False)

for idx in sample_indices:
    sample = dataset[idx]
    img_path = DATA_DIR / sample['psd_image']
    
    print(f"\nSample {idx}: {sample['num_carriers']} carriers")
    for carrier in sample['carriers']:
        print(f"  - {carrier['modulation']}: {carrier['symbol_rate_sps']/1e6:.1f} Msps, "
              f"BW={carrier['bandwidth_hz']/1e6:.1f} MHz, "
              f"FC={carrier['center_freq_hz']/1e6:.1f} MHz")
    
    # Display image
    from IPython.display import Image, display
    display(Image(filename=str(img_path)))

## Summary

This notebook uses GNURadio blocks to generate synthetic satellite downlink PSDs with labeled carriers. The dataset is saved to `data/raw/` with:
- PSD plots as PNG images
- PSD arrays as NumPy files
- Metadata labels in JSON format

Next steps:
1. Split data into train/val/test sets
2. Build ML model for carrier detection/classification
3. Train and evaluate model